# Week 4 Practical: Graph Neural Networks for Molecular Property Prediction
**AI for Drug Discovery**

## Overview

In this practical, you will:
1. **Represent molecules as graphs** — atoms become nodes, bonds become edges, and molecular properties become prediction targets
2. **Train a Graph Convolutional Network (GCN)** using DeepChem, a powerful open-source library for drug discovery ML
3. **Compare GNN performance to your Random Forest baseline** from Week 3 to see when deep learning helps
4. **Analyze when GNNs outperform classical ML** — and importantly, when they don't!

### Why Graph Neural Networks?

Traditional machine learning for molecules relies on **fixed fingerprints** (like ECFP/Morgan fingerprints) that encode molecular structure as bit vectors. While effective, these fingerprints:
- Are hand-designed and may miss important structural features
- Lose 3D spatial information
- Cannot be optimized for the specific prediction task

**Graph Neural Networks (GNNs)** take a fundamentally different approach: they **learn** molecular representations directly from the graph structure. The key innovation is **message passing** — an iterative process where each atom (node) collects information from its neighbors:

1. **Initialize**: Each atom starts with features (element type, charge, hybridization, etc.)
2. **Message**: Each atom sends a "message" to its neighbors — think of it like atoms gossiping about their local environment
3. **Aggregate**: Each atom collects messages from all neighbors and combines them (sum, mean, or max)
4. **Update**: Each atom updates its own representation based on collected messages
5. **Repeat**: Steps 2-4 repeat for K rounds. After K rounds, each atom "knows" about its K-hop neighborhood
6. **Readout**: All atom representations are pooled into a single molecular vector for prediction

**Intuitive analogy**: Imagine you're at a party and can only talk to people next to you. After round 1, you know about your immediate neighbors. After round 2, your neighbors tell you about *their* neighbors — so you know about people 2 steps away. After K rounds, you have information about everyone within K connections. This is exactly how GNNs work!

### Graph Representation of Molecules

A molecular graph consists of:
- **Adjacency Matrix (A)**: An N×N matrix where N is the number of atoms. A[i][j] = 1 if atoms i and j are bonded, 0 otherwise. This encodes the molecular topology.
- **Node Feature Matrix (X)**: An N×F matrix where F is the number of features per atom. Features include: atomic number, degree (number of bonds), formal charge, hybridization (sp, sp2, sp3), aromaticity, number of hydrogens, etc.
- **Edge Feature Matrix (E)**: Optionally, each bond can have features: bond type (single, double, triple, aromatic), conjugation, ring membership, stereochemistry (E/Z), etc.

For example, ethanol (C-C-O) has 3 heavy atoms, so its adjacency matrix is 3×3, and each atom has a feature vector describing its chemical properties.

### Prerequisites
- Completion of Weeks 1-3 (especially the RF model from Week 3)
- Basic understanding of neural networks (activation functions, backpropagation, loss functions)
- Python familiarity with numpy, matplotlib, and scikit-learn


In [ ]:
# ============================================================
# INSTALLATION AND SETUP
# Install all required Python packages for this practical.
# deepchem: provides GNN models and MoleculeNet benchmark datasets
# rdkit-pypi: the cheminformatics toolkit for molecular manipulation
# scikit-learn: classical ML library (for our RF baseline)
# pandas: data manipulation and DataFrames
# matplotlib & seaborn: plotting and visualization
# The -q flag suppresses verbose pip output for cleaner notebooks
# ============================================================
!pip install deepchem rdkit-pypi scikit-learn pandas matplotlib seaborn -q

# Suppress all warning messages to keep notebook output clean
# DeepChem and TensorFlow generate many deprecation warnings that
# are not relevant to our learning objectives
import warnings
warnings.filterwarnings('ignore')  # Filter out all warning messages


In [ ]:
# ============================================================
# IMPORT LIBRARIES
# Each library serves a specific purpose in our GNN pipeline.
# ============================================================

# deepchem: main library for molecular ML — provides datasets, featurizers,
# models (including GCN), and evaluation metrics all in one package
import deepchem as dc

# rdkit.Chem: core cheminformatics — parse SMILES strings into molecular objects,
# compute properties, and manipulate chemical structures
from rdkit import Chem

# AllChem: extended chemistry — includes 2D/3D coordinate generation and fingerprints
# Descriptors: compute molecular descriptors like MW, LogP, TPSA
# Draw: render molecules as 2D images for visualization
from rdkit.Chem import AllChem, Descriptors, Draw

# pandas: provides DataFrames for tabular data manipulation and display
import pandas as pd

# numpy: numerical computing — arrays, linear algebra, random number generation
import numpy as np

# matplotlib.pyplot: the foundational Python plotting library
import matplotlib.pyplot as plt

# seaborn: statistical visualization built on matplotlib — nicer defaults and plots
import seaborn as sns

# RandomForestClassifier: ensemble of decision trees — our classical ML baseline
# This is what we'll compare the GNN against
from sklearn.ensemble import RandomForestClassifier

# roc_auc_score: Area Under the Receiver Operating Characteristic curve
# Standard metric for binary classification — 0.5 = random, 1.0 = perfect
from sklearn.metrics import roc_auc_score

# Set seaborn's 'whitegrid' style for clean, publication-quality plots
# This adds a light grid background that helps read values off plots
sns.set_style('whitegrid')

# Print library version to confirm installation and for reproducibility
# Always record library versions in scientific work!
print(f'DeepChem version: {dc.__version__}')
print('All imports successful!')


## 1. Load a MoleculeNet Dataset

We'll use the **BACE** dataset: 1,513 compounds tested against **BACE-1** (Beta-site Amyloid precursor protein Cleaving Enzyme 1), a key target for Alzheimer's disease.

### Why BACE?
- BACE-1 cleaves amyloid precursor protein (APP) to produce amyloid-β peptides, which aggregate into plaques in Alzheimer's disease
- Inhibiting BACE-1 is a major therapeutic strategy (though clinical trials have been challenging)
- The dataset is a binary classification task: does a molecule inhibit BACE-1 (active=1) or not (active=0)?
- With ~1,500 molecules, it's small enough to train quickly but large enough to learn from

### How DeepChem Featurizes Molecules into Graphs
When we specify `featurizer='GraphConv'`, DeepChem converts each SMILES string into a graph:
1. **Parse SMILES** → RDKit molecular object
2. **Extract atoms** → Each atom becomes a node with features: atomic number, degree, formal charge, radical electrons, hybridization (one-hot: SP, SP2, SP3), aromaticity, total hydrogens
3. **Extract bonds** → Each bond becomes an edge with features: bond type (single/double/triple/aromatic), conjugation, ring membership
4. **Build graph** → Adjacency list + node feature matrix + edge feature matrix

### Scaffold Split
We use **scaffold splitting** instead of random splitting. This splits molecules based on their Murcko scaffolds (core ring systems), ensuring that the test set contains structurally novel molecules not seen during training. This is more realistic for drug discovery — you want to predict activity for *new* chemical series, not just interpolate within known series.

**Reference:** Wu, Z. et al. (2018). MoleculeNet: A Benchmark for Molecular Machine Learning. Chemical Science 9:513-530


In [ ]:
# ============================================================
# LOAD THE BACE DATASET WITH GRAPH FEATURIZATION
# This is the critical step where molecules become graphs.
# ============================================================

# dc.molnet.load_bace_classification() downloads and prepares the BACE dataset
# featurizer='GraphConv' tells DeepChem to convert each molecule into a graph
#   representation with atom-level node features and bond-level edge features
# splitter='scaffold' uses Murcko scaffold-based splitting for realistic evaluation
# Returns: task names, (train/valid/test) datasets, and any applied transformers
tasks, datasets, transformers = dc.molnet.load_bace_classification(
    featurizer='GraphConv',  # Convert SMILES to molecular graphs for GNN input
    splitter='scaffold'       # Split by molecular scaffold for realistic evaluation
)

# Unpack the three dataset splits from the returned tuple
# train_dataset: used to train the model (the model sees these molecules)
# valid_dataset: used to tune hyperparameters and monitor for overfitting
# test_dataset: held out completely — used only for final evaluation
train_dataset, valid_dataset, test_dataset = datasets

# Print dataset statistics to verify successful loading
# tasks tells us what we're predicting (BACE-1 inhibition)
print(f'Task: {tasks}')
# Print the size of each split to confirm reasonable proportions
# Typical split: ~80% train, ~10% valid, ~10% test
print(f'Training set: {len(train_dataset)} molecules')
print(f'Validation set: {len(valid_dataset)} molecules')
print(f'Test set: {len(test_dataset)} molecules')


## 2. Visualize Some Molecules from the Dataset

Before diving into modeling, it's essential to **look at your data**. Visualizing molecules helps you:
- Understand the chemical diversity in the dataset
- Spot potential issues (e.g., salts, fragments, very large molecules)
- Build chemical intuition about what makes a BACE-1 inhibitor

We'll render the first 12 molecules as 2D structure diagrams using RDKit's `Draw` module. Each molecule was originally stored as a SMILES (Simplified Molecular Input Line Entry System) string — a linear text representation of molecular structure.


In [ ]:
# ============================================================
# VISUALIZE MOLECULES FROM THE DATASET
# We need to reload with ECFP featurizer to access SMILES strings,
# because the GraphConv featurizer stores graph objects, not SMILES.
# ============================================================

# Load the same BACE dataset but with ECFP featurizer
# ECFP (Extended Connectivity Fingerprint) stores molecular IDs as SMILES strings
# which we need for visualization — the GraphConv featurizer doesn't preserve them
try:
    # Reload dataset with ECFP featurizer to access SMILES identifiers
    tasks2, datasets2, _ = dc.molnet.load_bace_classification(
        featurizer='ECFP', splitter='scaffold')  # Same scaffold split for consistency
    
    # Extract the first 12 SMILES strings from the training set
    # dataset.ids contains the molecule identifiers (SMILES strings)
    smiles_list = datasets2[0].ids[:12]
    
    # Convert SMILES strings to RDKit Mol objects for rendering
    # Chem.MolFromSmiles() parses SMILES; returns None if parsing fails
    # We filter out None values to handle any invalid SMILES gracefully
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s) is not None]
    
    # Render molecules in a grid layout: 4 molecules per row, each 300x250 pixels
    # MolsToGridImage creates a PIL image with 2D structure diagrams
    img = Draw.MolsToGridImage(mols[:12], molsPerRow=4, subImgSize=(300, 250))
    
    # Display the grid image in the Jupyter notebook
    display(img)
    
except Exception as e:
    # Gracefully handle errors (e.g., display not available in non-notebook environments)
    print(f'Visualization skipped: {e}')


## 3. Train a Graph Convolutional Network (GCN)

DeepChem's `GraphConvModel` implements a Graph Convolutional Network based on the architecture by Kipf & Welling (2017). Here's what happens under the hood:

### Architecture
1. **Input Layer**: Takes the molecular graph (atom features + adjacency matrix)
2. **Graph Convolution Layers**: Apply message passing — each atom aggregates information from its bonded neighbors using learned weight matrices. Mathematically: H^(l+1) = σ(D^(-1/2) A D^(-1/2) H^(l) W^(l)), where A is the adjacency matrix, D is the degree matrix, H is the node feature matrix, W is the learnable weight matrix, and σ is an activation function (ReLU)
3. **Graph Pooling (Readout)**: Aggregates all atom representations into a single molecular vector (using sum or mean pooling)
4. **Dense Layers**: Standard fully-connected layers that take the molecular vector and output a prediction

### Training Concepts
- **Epoch**: One complete pass through the entire training dataset. We train for 50 epochs, meaning the model sees every training molecule 50 times.
- **Loss Function**: Cross-entropy loss for binary classification — measures how far the model's predicted probabilities are from the true labels (0 or 1). Lower loss = better predictions.
- **Optimizer**: Adam optimizer (Adaptive Moment Estimation) — an advanced variant of stochastic gradient descent that adapts the learning rate for each parameter. It combines momentum (remembering past gradients) with RMSprop (scaling by running average of gradient magnitudes).
- **Learning Rate (0.001)**: Controls the step size during optimization. Too high → training is unstable and overshoots. Too low → training is slow and may get stuck. 0.001 is a common default.
- **Dropout (0.2)**: Randomly sets 20% of neuron outputs to zero during training. This is a regularization technique that prevents overfitting by forcing the network to learn redundant representations.
- **Batch Size (64)**: Number of molecules processed together before updating weights. Larger batches give more stable gradient estimates but use more memory.

**Reference:** Kipf, T.N. & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. ICLR


In [ ]:
# ============================================================
# BUILD AND TRAIN THE GRAPH CONVOLUTIONAL NETWORK (GCN)
# This is the core deep learning section of the practical.
# The GCN learns molecular representations directly from the
# graph structure rather than relying on hand-crafted fingerprints.
# ============================================================

# Instantiate a GraphConvModel with carefully chosen hyperparameters
# n_tasks=1: we have one prediction task (BACE-1 inhibition: yes/no)
# mode='classification': binary classification (not regression)
# dropout=0.2: randomly zero out 20% of neurons during training to prevent overfitting
# batch_size=64: process 64 molecules at a time (balances speed vs. memory)
# learning_rate=0.001: step size for the Adam optimizer (standard starting value)
model_gcn = dc.models.GraphConvModel(
    n_tasks=1,                # Single binary classification task
    mode='classification',    # Classification mode (outputs probabilities via sigmoid)
    dropout=0.2,              # 20% dropout for regularization
    batch_size=64,            # Number of molecules per training batch
    learning_rate=0.001       # Adam optimizer learning rate
)

# Train for 50 epochs, tracking loss at each epoch
# An epoch = one full pass through all training molecules
print('Training GCN...')

# List to store the loss value at each epoch for plotting the learning curve
losses = []

# Training loop: iterate for 50 epochs
for epoch in range(50):
    # model_gcn.fit() runs one epoch of training:
    #   1. Shuffle training data into batches of 64 molecules
    #   2. For each batch: forward pass → compute loss → backpropagate → update weights
    #   3. Return the average loss across all batches in this epoch
    loss = model_gcn.fit(train_dataset, nb_epoch=1)
    
    # Append this epoch's loss to our tracking list
    losses.append(loss)
    
    # Every 10 epochs, evaluate on both train and validation sets
    # This helps us monitor for overfitting (train AUC >> valid AUC)
    if (epoch + 1) % 10 == 0:
        # Evaluate using AUC-ROC metric on training set
        # AUC-ROC = Area Under Receiver Operating Characteristic curve
        train_score = model_gcn.evaluate(train_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
        
        # Evaluate on validation set (unseen during training)
        valid_score = model_gcn.evaluate(valid_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
        
        # Print progress: epoch number, training AUC, and validation AUC
        # Watch for: train AUC increasing while valid AUC plateaus/decreases = overfitting
        print(f'Epoch {epoch+1}: Train AUC={list(train_score.values())[0]:.3f}, '
              f'Valid AUC={list(valid_score.values())[0]:.3f}')

# Training complete — the model weights are now optimized
print('Training complete!')


In [ ]:
# ============================================================
# EVALUATE GCN ON THE HELD-OUT TEST SET
# This gives us the final, unbiased performance estimate.
# The model has NEVER seen these molecules during training.
# ============================================================

# Evaluate the trained GCN model on the test set using AUC-ROC
# dc.metrics.Metric wraps sklearn's roc_auc_score for use with DeepChem
# The test set was split by scaffold, so it contains structurally novel molecules
test_score_gcn = model_gcn.evaluate(test_dataset, [dc.metrics.Metric(dc.metrics.roc_auc_score)])

# Print the test AUC-ROC score
# Interpretation: 0.5 = random guessing, 0.7-0.8 = decent, 0.8-0.9 = good, >0.9 = excellent
print(f'GCN Test AUC-ROC: {list(test_score_gcn.values())[0]:.3f}')


## 4. Train a Random Forest Baseline

For a fair comparison, we train a **Random Forest** classifier using **Morgan fingerprints** (ECFP4). This is the classical ML approach from Week 3.

### Why Compare?
- GNNs are not always better! On small datasets (<5,000 molecules), classical methods like Random Forests with fingerprints often perform comparably or even better.
- Random Forests are faster to train, easier to interpret, and require less hyperparameter tuning.
- GNNs tend to shine on **larger datasets** (>10,000 molecules) where they can learn complex structure-activity relationships that fingerprints miss.

### Morgan Fingerprints vs. Learned GNN Features
- **Morgan/ECFP4**: Fixed-length binary vectors (2048 bits). Each bit indicates the presence/absence of a particular circular substructure (radius=2). Hand-designed and universal — the same fingerprint for all tasks.
- **GNN Features**: Learned real-valued vectors. The GNN learns which structural features matter *for this specific task*. Task-specific and adaptive, but requires more data to learn effectively.


In [ ]:
# ============================================================
# TRAIN A RANDOM FOREST BASELINE WITH MORGAN FINGERPRINTS
# This provides a classical ML comparison for the GCN.
# ============================================================

# Reload the BACE dataset with ECFP (Extended Connectivity Fingerprint) featurizer
# ECFP converts each molecule into a fixed-length binary vector (2048 bits by default)
# Each bit represents the presence/absence of a circular substructure at radius 2
# splitter='scaffold' ensures the SAME scaffold-based split as the GCN for fair comparison
tasks_ecfp, datasets_ecfp, transformers_ecfp = dc.molnet.load_bace_classification(
    featurizer='ECFP',       # Morgan/ECFP4 fingerprints as feature vectors
    splitter='scaffold'      # Same scaffold split as GCN for fair comparison
)

# Unpack the three splits (train, validation, test)
train_ecfp, valid_ecfp, test_ecfp = datasets_ecfp

# Initialize a Random Forest classifier with 500 decision trees
# n_estimators=500: number of trees in the forest (more trees = more stable predictions)
# random_state=42: fixed seed for reproducibility (same results every run)
# n_jobs=-1: use all available CPU cores for parallel training (speeds up significantly)
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)

# Train the Random Forest on fingerprint features (X) and binary labels (y)
# train_ecfp.X: 2D array of shape (n_molecules, 2048) — the fingerprint matrix
# train_ecfp.y.ravel(): 1D array of binary labels (0=inactive, 1=active)
# .ravel() flattens the array from shape (n,1) to (n,) as sklearn expects
rf.fit(train_ecfp.X, train_ecfp.y.ravel())

# Generate predicted probabilities for the positive class (active=1) on the test set
# predict_proba returns a 2D array: columns = [P(inactive), P(active)]
# We take [:, 1] to get the probability of being active (needed for AUC-ROC)
y_pred_rf = rf.predict_proba(test_ecfp.X)[:, 1]

# Compute AUC-ROC for the Random Forest predictions
# This measures how well the model ranks active molecules above inactive ones
auc_rf = roc_auc_score(test_ecfp.y.ravel(), y_pred_rf)

# Print the test AUC-ROC for comparison with the GCN
print(f'Random Forest Test AUC-ROC: {auc_rf:.3f}')


## 5. Compare Models

Now we compare the GCN and Random Forest side by side. Key considerations:

### When GNNs Outperform Random Forests
- **Large datasets** (>10,000 molecules): More data lets GNNs learn better representations
- **Complex SAR**: When structure-activity relationships are non-linear and depend on global molecular topology
- **Novel scaffolds**: GNNs may generalize better to entirely new chemical series
- **Multi-task learning**: GNNs can share learned representations across multiple related prediction tasks

### When Random Forests May Win
- **Small datasets** (<5,000 molecules): Not enough data for GNNs to learn meaningful representations
- **Well-characterized targets**: When hand-crafted features (e.g., specific pharmacophore patterns) capture the key SAR
- **Interpretability needed**: Random Forests allow feature importance analysis (which fingerprint bits matter)
- **Speed**: RF trains in seconds; GNNs may take minutes to hours on larger datasets

### Practical Advice
Always try both! In real drug discovery projects, ensemble methods that combine GNN and RF predictions often outperform either alone.


In [ ]:
# ============================================================
# COMPARE GCN vs RANDOM FOREST PERFORMANCE
# Side-by-side comparison of the two approaches.
# ============================================================

# Create a pandas DataFrame to display results in a clean table format
# Model names and their corresponding test AUC-ROC scores
results = pd.DataFrame({
    'Model': ['Random Forest (ECFP4)', 'Graph Conv Network'],  # Model names for the table
    'Test AUC-ROC': [auc_rf, list(test_score_gcn.values())[0]]  # Their test scores
})

# Print the results table (index=False hides the row numbers for cleaner output)
print(results.to_string(index=False))
print()  # Blank line for readability

# Compare the two models and print an interpretation
# This teaches students that GNNs don't always win!
if list(test_score_gcn.values())[0] > auc_rf:
    # GCN has higher AUC — it learned useful graph-level features
    print('GCN outperforms RF on this dataset!')
else:
    # RF is competitive — this is expected for small datasets like BACE
    print('RF is competitive with (or better than) GCN on this dataset.')
    # Explain WHY this happens — it's an important lesson
    print('This is common for smaller datasets like BACE (~1500 molecules).')


In [ ]:
# ============================================================
# PLOT THE GCN TRAINING LOSS CURVE
# The loss curve shows how well the model is learning over time.
# A decreasing curve indicates the model is fitting the training data.
# ============================================================

# Create a figure with specified dimensions (width=10, height=5 inches)
plt.figure(figsize=(10, 5))

# Plot the loss values (y-axis) against epoch numbers (x-axis, implicit)
# color='#1f77b4': matplotlib's default blue — professional and accessible
# linewidth=1: thin line to show detail in the loss curve fluctuations
plt.plot(losses, color='#1f77b4', linewidth=1)

# Label the x-axis as 'Epoch' (one pass through the training data)
plt.xlabel('Epoch', fontsize=12)

# Label the y-axis as 'Training Loss' (cross-entropy loss value)
plt.ylabel('Training Loss', fontsize=12)

# Add a descriptive title to the plot
plt.title('GCN Training Loss', fontsize=14)

# Adjust layout to prevent labels from being cut off
plt.tight_layout()

# Render and display the plot in the notebook
plt.show()


## 6. (Bonus) Try the HIV Dataset

The **HIV dataset** has ~41,000 molecules — significantly larger than BACE. This is where GNNs should have a **bigger advantage** over fingerprint-based methods, because:
- More training data allows the GNN to learn richer, more nuanced molecular representations
- The chemical diversity is greater, so learned features can capture more complex SAR patterns
- Morgan fingerprints have a fixed vocabulary of substructures, while GNNs can adapt their feature extraction to the specific task

**Note:** Training on the HIV dataset takes significantly longer (5-15 minutes depending on your hardware). Uncomment the code below to try it.


In [ ]:
# ============================================================
# (BONUS) HIV DATASET — LARGER DATASET WHERE GNNs SHINE
# Uncomment this entire block to run the HIV experiment.
# The HIV dataset has ~41,000 molecules — much larger than BACE.
# With more data, GNNs can learn more expressive representations
# and typically outperform fingerprint-based methods.
# ============================================================

# # Load the HIV dataset with graph featurization and scaffold split
# # This dataset classifies molecules as active/inactive against HIV replication
# tasks_hiv, datasets_hiv, _ = dc.molnet.load_hiv(
#     featurizer='GraphConv', splitter='scaffold')  # Same setup as BACE
#
# # Unpack train/valid/test splits
# train_hiv, valid_hiv, test_hiv = datasets_hiv
#
# # Print dataset sizes to verify loading
# print(f'HIV dataset: {len(train_hiv)} train, {len(test_hiv)} test')
#
# # Build a GCN model with the same architecture as before
# # Same hyperparameters for fair comparison
# model_hiv = dc.models.GraphConvModel(n_tasks=1, mode='classification',
#                                       dropout=0.2, learning_rate=0.001)
#
# # Train for 30 epochs (fewer than BACE because the dataset is larger)
# # Each epoch processes ~41,000 molecules, so training takes longer
# model_hiv.fit(train_hiv, nb_epoch=30)
#
# # Evaluate on the held-out test set
# score_hiv = model_hiv.evaluate(test_hiv, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
#
# # Print the test AUC — compare this to an RF baseline on the same data
# print(f'HIV GCN Test AUC: {list(score_hiv.values())[0]:.3f}')


## SpikerBot Neural Motifs: Physiology as Graphs

### Cross-Domain Insight: From Molecular Graphs to Brain Graphs

Everything you've learned about Graph Neural Networks for molecules applies **directly** to neuroscience data! The mathematical framework is identical — only the domain interpretation changes:

| Molecular Graphs | Neural Graphs |
|---|---|
| **Nodes** = atoms (C, N, O, ...) | **Nodes** = recording channels/electrodes |
| **Node features** = atomic number, charge, hybridization | **Node features** = firing rate, CV of ISI, mean amplitude |
| **Edges** = chemical bonds | **Edges** = correlations between electrode pairs |
| **Edge features** = bond type, conjugation | **Edge features** = correlation strength, time lag |
| **Graph-level task** = predict molecular property | **Graph-level task** = classify brain state or predict drug effect |

### How Neural Recording Data Becomes a Graph

When you record from the brain (or from a programmable **SpikerBot** neural stimulator), you typically have multiple recording channels. Each channel records voltage over time, producing **spike trains** — sequences of action potentials. To build a graph:

1. **Nodes**: Each recording electrode/channel becomes a node
2. **Node features**: Compute spike train statistics for each channel — firing rate, coefficient of variation of inter-spike intervals (CV-ISI), burst index, mean amplitude, etc.
3. **Edges**: Compute pairwise correlations between channels (cross-correlation, coherence, Granger causality). If the correlation exceeds a threshold, draw an edge between those two nodes.
4. **Edge features**: The strength and timing of correlations (e.g., correlation coefficient, phase lag)
5. **Feed the graph to a GNN**: The SAME message passing framework you used for molecules now classifies brain states, predicts drug effects, or detects pathological activity

### Programmable SpikerBot Neural Motifs

The **SpikerBot** is a programmable neural stimulator (Marzullo & Gage, 2012) that can generate three fundamental firing patterns (motifs):

1. **Tonic Firing** (regular, evenly-spaced spikes)
   - Found in: motor neurons during sustained contraction, cerebellar Purkinje cells at rest
   - Characterized by: very low CV-ISI (~0), constant firing rate
   - On SpikerBot: constant-frequency stimulation (e.g., every 50 ms = 20 Hz)
   - Drug relevance: Na+ channel blockers (lidocaine) decrease tonic firing rate

2. **Burst Firing** (clusters of rapid spikes separated by pauses)
   - Found in: thalamocortical neurons during sleep spindles, dopamine neurons during reward, hippocampal place cells
   - Characterized by: high CV-ISI (>1), bimodal ISI distribution (short intra-burst, long inter-burst)
   - On SpikerBot: 3-4 rapid pulses at 100 Hz, then 200-400 ms pause, repeat
   - Drug relevance: T-type Ca2+ channel blockers (ethosuximide) suppress burst firing in absence epilepsy

3. **Irregular/Poisson Firing** (random intervals)
   - Found in: cortical neurons during wakefulness, most brain areas under active conditions
   - Characterized by: CV-ISI ≈ 1 (hallmark of a Poisson process), exponentially distributed ISIs
   - On SpikerBot: randomized stimulation intervals drawn from an exponential distribution
   - Drug relevance: GABA-A agonists (benzodiazepines) can reduce firing rate and alter regularity

### Spike Train Statistics as Node Features

Just as we compute molecular descriptors (MW, LogP, TPSA) to characterize molecules, we compute **spike train statistics** to characterize neural activity:

- **Firing Rate** (Hz): Total spikes / recording duration — analogous to molecular weight
- **CV of ISI** (Coefficient of Variation): std(ISI) / mean(ISI) — the key statistic that distinguishes firing patterns:
  - CV ≈ 0: perfectly regular (tonic)
  - CV ≈ 1: random (Poisson/irregular)
  - CV > 1: bursty (more variable than random)
- **Burst Index**: Fraction of spikes occurring in bursts — analogous to aromaticity index
- **Mean Amplitude**: Average spike height — reflects recording quality and cell type

### GNNs for Brain Connectivity Data

Recent work has applied GNNs to brain connectivity graphs with great success:
- **Bessadok, A., Mahjoub, M.A., & Rekik, I. (2022)**. Graph Neural Networks in Network Neuroscience. *Medical Image Analysis*, 79, 102418. — Comprehensive review of GNN applications in brain connectivity analysis, including disease classification, brain age prediction, and connectome analysis.

### The Key Takeaway

**Students who learn molecular GNNs can immediately work in computational neuroscience** — and vice versa! The mathematical framework (message passing on graphs) is domain-agnostic. The only difference is what the nodes, edges, and features represent. This is the power of graph-based thinking: it provides a unified computational framework across biology.

**Reference:** Marzullo, T.C. & Gage, G.J. (2012). The SpikerBox: A Low Cost, Open-Source BioAmplifier for Increasing Public Participation in Neuroscience Inquiry. *Advances in Physiology Education*, 36(2), 2-14.


In [ ]:
# ============================================================
# PROGRAMMABLE SPIKERBOT NEURAL MOTIFS
# This demonstrates how neural firing patterns can be represented
# and analyzed with the same computational tools as molecular data.
# The SpikerBot can be programmed to generate these motifs.
# Reference: Marzullo & Gage (2012), Adv. Physiol. Educ. 36:2-14
# ============================================================

# numpy: numerical computing library for array operations and random number generation
import numpy as np

# matplotlib.pyplot: plotting library for creating spike raster visualizations
import matplotlib.pyplot as plt

# Set random seed for reproducibility
# This ensures the same random spike trains are generated every time
np.random.seed(42)

# --- Simulate three neural firing motifs ---
# These are the patterns you can program the SpikerBot to generate
# and observe the muscle/neural response

# Total simulation time in seconds
T = 2.0  # 2 seconds of recording

# Time resolution: 1 ms (0.001 s), which is 1000 Hz sampling rate
# This is typical for neural data acquisition systems
dt = 0.001  # 1 ms time resolution (1000 Hz sampling, typical for neural data)

# Create the time array from 0 to T in steps of dt
# This gives us 2000 time points for our 2-second simulation
t = np.arange(0, T, dt)  # Time array

# MOTIF 1: TONIC FIRING (regular, evenly-spaced spikes)
# Found in: motor neurons during sustained contraction, cerebellar Purkinje cells
# On SpikerBot: constant-frequency stimulation (e.g., every 50 ms = 20 Hz)
# Drug effect: reducing Na+ conductance (lidocaine) decreases firing rate

# Set the firing rate to 20 Hz (20 spikes per second)
tonic_rate = 20  # Hz (spikes per second)

# Calculate the Inter-Spike Interval (time between consecutive spikes)
# ISI = 1/rate = 1/20 = 0.05 seconds = 50 ms
tonic_isi = 1.0 / tonic_rate  # Inter-Spike Interval in seconds (50 ms)

# Generate perfectly regular spike times using np.arange
# Spikes occur at 50ms, 100ms, 150ms, ..., up to 2000ms
tonic_spike_times = np.arange(tonic_isi, T, tonic_isi)  # Regular spike times

# MOTIF 2: BURST FIRING (clusters of rapid spikes with pauses)
# Found in: thalamocortical neurons (sleep spindles), dopamine neurons (reward), hippocampus
# On SpikerBot: 3-4 rapid pulses at 100 Hz, then 300 ms pause, repeat
# Drug effect: T-type Ca2+ channel blockers (ethosuximide) suppress burst firing

# Initialize empty list to collect burst spike times
burst_spike_times = []

# Start the first burst at 100 ms into the recording
burst_time = 0.1  # Start time

# Generate bursts until we exceed the simulation duration
while burst_time < T:
    # Generate a burst: 3-4 spikes at 100 Hz (10 ms apart)
    # np.random.randint(3, 5) gives either 3 or 4 spikes per burst
    n_spikes_in_burst = np.random.randint(3, 5)  # 3-4 spikes per burst
    
    # Add each spike in the burst, separated by 10 ms (100 Hz intra-burst frequency)
    for j in range(n_spikes_in_burst):
        burst_spike_times.append(burst_time + j * 0.01)  # 10 ms ISI within burst
    
    # Add inter-burst interval: 200-400 ms pause between bursts
    # This creates the characteristic burst-pause-burst-pause pattern
    burst_time += n_spikes_in_burst * 0.01 + np.random.uniform(0.2, 0.4)

# Convert to numpy array and clip to simulation duration
# Remove any spikes that fall after our 2-second window
burst_spike_times = np.array([s for s in burst_spike_times if s < T])

# MOTIF 3: IRREGULAR / POISSON FIRING (random intervals)
# Found in: cortical neurons during wakefulness, most brain areas
# On SpikerBot: randomized stimulation intervals from exponential distribution
# The Poisson process is the simplest model of "random" neural firing
# Drug effect: GABA-A agonists (benzodiazepines) can make firing more irregular/slower

# Average rate of 20 Hz (same as tonic, for fair comparison of statistics)
irregular_rate = 20  # Average rate (Hz) - same as tonic for fair comparison

# Generate inter-spike intervals from an exponential distribution
# For a Poisson process, ISIs are exponentially distributed with mean = 1/rate
# Generate 100 ISIs (more than enough for 2 seconds at 20 Hz)
irregular_isis = np.random.exponential(1.0 / irregular_rate, size=100)  # Exponential ISIs

# Convert ISIs to spike times using cumulative sum
# If ISIs are [0.03, 0.07, 0.02, ...], spike times are [0.03, 0.10, 0.12, ...]
irregular_spike_times = np.cumsum(irregular_isis)  # Cumulative sum gives spike times

# Clip spike times to our simulation window [0, T)
irregular_spike_times = irregular_spike_times[irregular_spike_times < T]  # Clip to T

# --- Create spike raster plots ---
# Create a figure with 3 vertically-stacked subplots sharing the x-axis
# figsize=(14, 8): wide enough to see individual spikes, tall enough for 3 panels
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Helper function to create a spike train voltage trace from spike times
# This converts a list of spike times into a binary time series for plotting
def spikes_to_trace(spike_times, t, spike_height=1.0, spike_width=0.002):
    # Initialize an array of zeros (no spikes) with the same length as the time array
    trace = np.zeros_like(t)
    # For each spike time, set the trace to spike_height in a small window around it
    for st in spike_times:
        # Create a boolean mask: True for time points within spike_width of the spike time
        mask = np.abs(t - st) < spike_width
        # Set those time points to the spike height (creates a narrow rectangular pulse)
        trace[mask] = spike_height
    # Return the complete trace array
    return trace

# Plot tonic firing pattern (blue, top panel)
# Convert spike times to a plottable voltage trace
trace_tonic = spikes_to_trace(tonic_spike_times, t)
# Plot the trace: time in milliseconds on x-axis, spikes on y-axis
axes[0].plot(t * 1000, trace_tonic, 'b-', linewidth=0.8)
# Label the y-axis
axes[0].set_ylabel('Spikes', fontsize=11)
# Add a descriptive title explaining where this pattern is found
axes[0].set_title('TONIC FIRING - Regular spikes (motor neurons, pacemaker cells)', fontsize=13, fontweight='bold')
# Set y-axis limits to give some padding above and below the spikes
axes[0].set_ylim(-0.1, 1.3)

# Plot burst firing pattern (red, middle panel)
# Convert burst spike times to a plottable voltage trace
trace_burst = spikes_to_trace(burst_spike_times, t)
# Plot in red to visually distinguish from tonic pattern
axes[1].plot(t * 1000, trace_burst, 'r-', linewidth=0.8)
# Label the y-axis
axes[1].set_ylabel('Spikes', fontsize=11)
# Add a descriptive title explaining where this pattern is found
axes[1].set_title('BURST FIRING - Clustered spikes (thalamic, dopamine neurons)', fontsize=13, fontweight='bold')
# Set y-axis limits for consistent scaling across panels
axes[1].set_ylim(-0.1, 1.3)

# Plot irregular firing pattern (green, bottom panel)
# Convert irregular spike times to a plottable voltage trace
trace_irreg = spikes_to_trace(irregular_spike_times, t)
# Plot in green to visually distinguish from other patterns
axes[2].plot(t * 1000, trace_irreg, 'g-', linewidth=0.8)
# Label the y-axis
axes[2].set_ylabel('Spikes', fontsize=11)
# Label the x-axis (only on bottom panel since they share x-axis)
axes[2].set_xlabel('Time (ms)', fontsize=12)
# Add a descriptive title explaining where this pattern is found
axes[2].set_title('IRREGULAR (POISSON) FIRING - Random intervals (cortical neurons)', fontsize=13, fontweight='bold')
# Set y-axis limits for consistent scaling across panels
axes[2].set_ylim(-0.1, 1.3)

# Add a super-title above all three panels explaining the overall figure
plt.suptitle('Programmable SpikerBot Neural Motifs\nThese patterns can be generated by programming the SpikerBot stimulator', fontsize=14, y=1.02)

# Adjust subplot spacing to prevent overlapping labels
plt.tight_layout()

# Render and display the figure in the notebook
plt.show()

# --- Compute spike train statistics ---
# These features are analogous to molecular descriptors!
# Just as we compute MW, LogP, TPSA for molecules,
# we compute firing rate, CV(ISI), burst index for spike trains.

# Define a function to compute and display spike train statistics
# This function takes spike times and a descriptive name as input
def compute_spike_stats(spike_times, name):
    # Inter-Spike Intervals: time between consecutive spikes
    # Need at least 2 spikes to compute intervals
    if len(spike_times) < 2:
        print(f"{name}: Too few spikes for statistics")
        return
    
    # Compute ISIs using np.diff (differences between consecutive elements)
    # If spike times are [0.05, 0.10, 0.15], ISIs are [0.05, 0.05]
    isis = np.diff(spike_times)  # Differences between consecutive spike times
    
    # Mean firing rate: total number of spikes divided by recording duration
    # Units: Hz (spikes per second)
    firing_rate = len(spike_times) / T
    
    # Mean ISI: average time between consecutive spikes
    # Convert from seconds to milliseconds (* 1000) for more intuitive display
    mean_isi = np.mean(isis) * 1000  # Convert to ms
    
    # CV of ISI (Coefficient of Variation): std(ISI) / mean(ISI)
    # This is the KEY statistic that distinguishes firing patterns:
    # CV = 0: perfectly regular (tonic) — all intervals are identical
    # CV = 1: Poisson (random) — intervals follow exponential distribution
    # CV > 1: bursty (more variable than random) — mix of short and long intervals
    cv_isi = np.std(isis) / np.mean(isis)
    
    # Print all computed statistics in a formatted display
    print(f"  {name}:")
    print(f"    Spike count: {len(spike_times)}")
    print(f"    Firing rate: {firing_rate:.1f} Hz")
    print(f"    Mean ISI: {mean_isi:.1f} ms")
    # Print CV-ISI with an interpretation label
    print(f"    CV(ISI): {cv_isi:.3f}  {'(regular)' if cv_isi < 0.3 else '(irregular)' if cv_isi < 0.8 else '(bursty/very irregular)'}")

# Print header for the statistics section
print("\n=== SPIKE TRAIN STATISTICS (analogous to molecular descriptors) ===")

# Compute and display statistics for each of the three motifs
# Compare the CV-ISI values across motifs — this is the key distinguishing feature
compute_spike_stats(tonic_spike_times, "Tonic")
compute_spike_stats(burst_spike_times, "Burst")
compute_spike_stats(irregular_spike_times, "Irregular/Poisson")

# Print the connection between neural graphs and molecular graphs
# This reinforces the cross-domain insight that is the key takeaway
print("\n=== CONNECTION TO GNNs ===")
print("In a multi-electrode recording (e.g., SpikerBot with multiple channels):")
print("  Nodes = electrodes/channels")
print("  Node features = [firing_rate, CV_ISI, mean_amplitude, ...]")
print("  Edges = correlations between channels")
print("  -> Feed to GNN -> Classify brain state or predict drug effect")
print("\nThis is EXACTLY the same framework as molecular GNNs!")
print("  Molecular: Nodes=atoms, Edges=bonds, Task=predict toxicity")
print("  Neural: Nodes=electrodes, Edges=correlations, Task=classify brain state")


## 7. Exercises

1. **Hyperparameter tuning**: Try different learning rates (0.0001, 0.001, 0.01), dropout rates (0.1, 0.2, 0.5), and number of epochs (20, 50, 100). Record the validation AUC for each configuration. Which combination works best?
2. **Deeper model**: Add more graph conv layers by exploring DeepChem's model options. Does a deeper model help? (Hint: very deep GNNs can suffer from "over-smoothing" where all node representations become similar)
3. **Different dataset**: Try the Tox21 dataset (`dc.molnet.load_tox21`) — this is a multi-task dataset with 12 toxicity endpoints. How does GCN performance vary across tasks?
4. **Chemprop**: Install and try Chemprop (`pip install chemprop`) — a message-passing neural network that is often state-of-the-art. How does it compare to DeepChem's GCN?


## References
- Wu, Z. et al. (2018). MoleculeNet: A Benchmark for Molecular Machine Learning. Chemical Science 9:513-530
- Kipf, T.N. & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. ICLR
- Yang, K. et al. (2019). Analyzing Learned Molecular Representations for Property Prediction. JCIM 59:3370-3388
- Stokes, J.M. et al. (2020). A Deep Learning Approach to Antibiotic Discovery. Cell 180:688-702
- Marzullo, T.C. & Gage, G.J. (2012). The SpikerBox: A Low Cost, Open-Source BioAmplifier for Increasing Public Participation in Neuroscience Inquiry. Advances in Physiology Education 36(2):2-14
- Bessadok, A., Mahjoub, M.A., & Rekik, I. (2022). Graph Neural Networks in Network Neuroscience. Medical Image Analysis 79:102418
